# Cat vs Dog Classifier — Solving Underfitting with a Balanced Basic CNN

This notebook is a **separate teaching notebook** focused on the problem of **underfitting**.

Previously, we tried to reduce overfitting using strong techniques such as:

- Heavy Dropout
- Strong L2 Regularization
- GlobalAveragePooling2D
- Data Augmentation
- EarlyStopping

But sometimes, if we use too much regularization, the model becomes too weak.  
Then it may fail to learn important features of cats and dogs.

That problem is called **underfitting**.

In this notebook, we will:

1. Download the Cat vs Dog dataset from Kaggle.
2. Prepare training and validation data.
3. Diagnose underfitting.
4. Build a balanced basic CNN model.
5. Train the model with moderate regularization.
6. Plot accuracy and loss graphs.
7. Test predictions on uploaded images.


# 1. What is Underfitting?

Underfitting happens when a model performs poorly on both training data and validation data.

Example:

| Training Accuracy | Validation Accuracy | Meaning |
|---|---|---|
| 55% | 54% | Severe underfitting |
| 65% | 63% | Model is too weak |
| 85% | 82% | Good learning |
| 97% | 72% | Overfitting |

In simple words:

> Underfitting means the model has not learned enough from the training data.

For Cat vs Dog classification, an underfit model may confuse clear dog images as cats or clear cat images as dogs.


# 2. Difference Between Overfitting and Underfitting

| Problem | Training Accuracy | Validation Accuracy | Meaning |
|---|---|---|---|
| Underfitting | Low | Low | Model did not learn enough |
| Good Fit | Good | Good | Model learned useful patterns |
| Overfitting | Very High | Low | Model memorized training data |

## Teaching Example

Overfitting is like a student who memorizes textbook answers but fails in new questions.

Underfitting is like a student who did not study enough and performs poorly everywhere.


# 3. Install Kaggle API

In [ ]:
!pip install -q --upgrade kaggle

# 4. Load Kaggle Credentials from Colab Secrets

Before running this cell, add your Kaggle credentials in Colab Secrets.

Go to:

**Colab left sidebar → Key icon → Add new secret**

Add these two secrets:

```text
KAGGLE_USERNAME
KAGGLE_KEY
```

This is safer than uploading `kaggle.json` directly.


In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded successfully.")

# 5. Download Microsoft Cats vs Dogs Dataset from Kaggle

In [ ]:
!mkdir -p /content/data
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data

# 6. Unzip Dataset

In [ ]:
import zipfile
import os

zip_path = "/content/data/microsoft-catsvsdogs-dataset.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

# 7. Check Dataset Structure

The dataset should contain two folders:

```text
PetImages/
    Cat/
    Dog/
```


In [ ]:
base_dir = "/content/data/PetImages"

print("Available folders:", os.listdir(base_dir))
print("Cat images:", len(os.listdir(os.path.join(base_dir, "Cat"))))
print("Dog images:", len(os.listdir(os.path.join(base_dir, "Dog"))))

# 8. Remove Corrupted Images

Some images in this dataset are corrupted.  
If we do not remove them, training may stop with an error.


In [ ]:
from PIL import Image

classes = ["Cat", "Dog"]
removed = 0

for class_name in classes:
    folder_path = os.path.join(base_dir, class_name)

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)

        try:
            img = Image.open(img_path)
            img.verify()
        except Exception:
            os.remove(img_path)
            removed += 1

print("Corrupted images removed:", removed)

# 9. Create Train and Validation Data

## Important Change for Underfitting

In the previous overfitting solution, augmentation and regularization were strong.

To solve underfitting, we use **moderate augmentation** instead of very heavy augmentation.

Why?

Because too much augmentation can make the learning task too difficult for a basic CNN.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 150
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    # Moderate augmentation
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

print("Class labels:", train_data.class_indices)

# 10. Visualize Training Images

Always visualize images before training.

This helps us check:

- images are loading correctly
- labels are correct
- augmentation is not too strong


In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_data)

plt.figure(figsize=(10, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title("Dog" if labels[i] == 1 else "Cat")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 11. How to Diagnose Underfitting

After training, underfitting usually appears like this:

| Symptom | Meaning |
|---|---|
| Training accuracy is low | Model is not learning enough |
| Validation accuracy is also low | Model generalization is also weak |
| Training loss is high | Model predictions are poor |
| Both training and validation curves are poor | Model capacity may be too low |

Possible solutions:

1. Use a slightly larger CNN.
2. Reduce excessive Dropout.
3. Reduce excessive L2 regularization.
4. Train for more epochs.
5. Use a better learning rate.
6. Use moderate augmentation instead of heavy augmentation.


# 12. Build Balanced Basic CNN Model

This model is still a **basic CNN**, not transfer learning.

But it is stronger than the previous over-regularized model.

## Main Improvements

| Previous Problem | New Solution |
|---|---|
| Too much Dropout | Moderate Dropout |
| Strong L2 regularization | Light L2 regularization |
| GlobalAveragePooling2D reduced learning too much | Use Flatten for stronger learning |
| Too much augmentation | Moderate augmentation |
| Model too weak | More balanced CNN capacity |


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.regularizers import l2

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", padding="same",
           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(256, activation="relu", kernel_regularizer=l2(0.0001)),
    Dropout(0.4),

    Dense(1, activation="sigmoid")
])

model.summary()

# 13. Why This Model Helps Underfitting

## Conv2D Layers

Convolution layers learn visual features such as:

- edges
- curves
- eyes
- nose
- ears
- fur texture
- face shape

## BatchNormalization

It makes training more stable.

## Flatten

Flatten gives the classifier more information compared to GlobalAveragePooling2D.  
This can improve learning when the previous model was too weak.

## Dense Layer with 256 Neurons

This gives the model more learning power.

## Dropout 0.4

This is moderate.  
It reduces overfitting but does not make the model too weak.

## L2 = 0.0001

This is light regularization.  
It prevents very large weights but does not heavily punish the model.


# 14. Compile the Model

We use Adam optimizer with a moderate learning rate.

If the learning rate is too high, training becomes unstable.  
If the learning rate is too low, learning becomes very slow.


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 15. Train the Model

We still use callbacks, but with balanced settings.

## EarlyStopping

Stops training if validation loss does not improve.

## ReduceLROnPlateau

Reduces learning rate when the model gets stuck.

## ModelCheckpoint

Saves the best model automatically.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7
    ),

    ModelCheckpoint(
        "/content/best_underfit_solution_basic_cnn.h5",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    )
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=40,
    callbacks=callbacks
)

# 16. Plot Accuracy Graph

## How to Read This Graph

### Underfitting
Both training accuracy and validation accuracy remain low.

### Good Fit
Training and validation accuracy both improve and stay close.

### Overfitting
Training accuracy becomes very high, but validation accuracy stays low or drops.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

# 17. Plot Loss Graph

## How to Read This Graph

### Underfitting
Both training loss and validation loss remain high.

### Good Fit
Both losses decrease and stay close.

### Overfitting
Training loss decreases, but validation loss increases.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# 18. Evaluate Best Model

In [ ]:
loss, accuracy = model.evaluate(val_data)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

# 19. Load the Best Saved Model

The best model may not always be the final epoch model.  
So we load the model saved by ModelCheckpoint.


In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model("/content/best_underfit_solution_basic_cnn.h5")

print("Best underfitting-solution model loaded successfully.")

# 20. Test on Random Dog Images from Dataset

This is a very useful diagnostic step.

If the model predicts many real dog images as cats, then the model still has a learning problem.


In [ ]:
import random
import numpy as np
from tensorflow.keras.preprocessing import image

dog_folder = "/content/data/PetImages/Dog"

dog_images = random.sample(os.listdir(dog_folder), 5)

print("Class mapping:", train_data.class_indices)

for img_name in dog_images:
    img_path = os.path.join(dog_folder, img_name)

    try:
        img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img)
        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        score = best_model.predict(img_array)[0][0]

        plt.imshow(img)
        plt.axis("off")
        plt.show()

        print("Image:", img_name)
        print("Raw score:", score)

        if score >= 0.5:
            print("Prediction: Dog")
        else:
            print("Prediction: Cat")

        print("-" * 40)

    except Exception as e:
        print("Skipped image:", img_name, "Error:", e)

# 21. Test on Random Cat Images from Dataset

This checks whether the model can correctly identify cats.


In [ ]:
cat_folder = "/content/data/PetImages/Cat"

cat_images = random.sample(os.listdir(cat_folder), 5)

print("Class mapping:", train_data.class_indices)

for img_name in cat_images:
    img_path = os.path.join(cat_folder, img_name)

    try:
        img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img)
        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        score = best_model.predict(img_array)[0][0]

        plt.imshow(img)
        plt.axis("off")
        plt.show()

        print("Image:", img_name)
        print("Raw score:", score)

        if score >= 0.5:
            print("Prediction: Dog")
        else:
            print("Prediction: Cat")

        print("-" * 40)

    except Exception as e:
        print("Skipped image:", img_name, "Error:", e)

# 22. Upload Your Own Image and Predict

This is the final prediction cell.

It uses a safer confidence threshold:

- Score >= 0.65 → Dog
- Score <= 0.35 → Cat
- Between 0.35 and 0.65 → Uncertain

This avoids giving a strong answer when the model is unsure.


In [ ]:
from google.colab import files

print("Class mapping:", train_data.class_indices)

uploaded = files.upload()

for file_name in uploaded.keys():
    img_path = file_name

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    score = best_model.predict(img_array)[0][0]

    plt.imshow(img)
    plt.axis("off")
    plt.show()

    print("Raw prediction score:", score)

    if score >= 0.65:
        print("Prediction: Dog")
        print("Confidence:", round(float(score) * 100, 2), "%")

    elif score <= 0.35:
        print("Prediction: Cat")
        print("Confidence:", round(float(1 - score) * 100, 2), "%")

    else:
        print("Prediction: Uncertain")
        print("The model is not confident enough.")

# 23. What If It Still Predicts Wrong?

If the model still predicts a clear dog as cat or a clear cat as dog, check these points:

## 1. Check Validation Accuracy

If validation accuracy is low, the model has not learned well.

## 2. Check Class Mapping

Run:

```python
print(train_data.class_indices)
```

Usually:

```text
{'Cat': 0, 'Dog': 1}
```

For sigmoid output:

- score close to 0 means class 0
- score close to 1 means class 1

## 3. Check Uploaded Image

The model may fail if the uploaded image has:

- very small animal
- unusual pose
- cartoon or AI-generated image
- too much background
- poor lighting
- multiple animals

## 4. Basic CNN Limitation

A basic CNN is for learning concepts.  
For professional results, use transfer learning:

- MobileNetV2
- EfficientNet
- ResNet50
- VGG16


# 24. Lecture Summary

In this notebook, we solved the problem of **underfitting**.

We learned that:

- Overfitting means the model memorizes training data.
- Underfitting means the model does not learn enough.
- Too much regularization can cause underfitting.
- A balanced CNN needs enough learning power and moderate regularization.

## Final Teaching Message

A good deep learning model needs balance.

If the model is too complex, it may overfit.  
If the model is too weak, it may underfit.

The goal is to find the right balance between learning power and generalization.
